In [ ]:
from google.colab import files
import pandas as pd
import io
import numpy as np
import matplotlib.pyplot as plt

uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]), nrows=50000)

print(df.head())
print(df.shape)

In [ ]:

from IPython.display import display, html

df_jan = pd.read_csv('2020-Jan.csv')
df_feb = pd.read_csv('2020-Feb.csv')

df_jan_clean = df_jan.dropna(subset=['user_id', 'event_type', 'event_time']).drop_duplicates()
df_feb_clean = df_feb.dropna(subset=['user_id', 'event_type', 'event_time']).drop_duplicates()

In [ ]:
# January and February sales

display(HTML('<h1>---January---</h1>'))
display(HTML('<div style="height: 5px;"></div>'))
display(df_jan_clean.head())

display(HTML('<div style ="height: 15px"></div>'))

display(HTML('<h1>---February---</h1>'))
display(HTML('<div style = "height: 5px"></div>'))
display(df_feb_clean.head())

In [ ]:
# calculating cart rate, drop off cart, purchase rate, drop off purchase, overall rate according to January and February months

jan_s = df_jan_clean.groupby('event_type')['user_id'].nunique()

jan_s['cart_rate_%'] = (jan_s['cart'] / jan_s['view']) * 100
jan_s['drop_off_cart_%'] = 100 - jan_s['cart_rate_%']
jan_s['purchase_rate_%'] = (jan_s['purchase'] / jan_s['cart']) * 100
jan_s['drop_off_purchase_%'] = 100 - jan_s['purchase_rate_%']
jan_s['overall_rate_%'] = (jan_s['purchase'] / jan_s['view']) * 100

print("--- January ---")
print(jan_s)

feb_s = df_feb_clean.groupby('event_type')['user_id'].nunique()

feb_s['cart_rate_%'] = (feb_s['cart'] / feb_s['view']) * 100
feb_s['drop_off_cart_%'] = 100 - feb_s['cart_rate_%']
feb_s['purchase_rate_%'] = (feb_s['purchase'] / feb_s['cart']) * 100
feb_s['drop_off_purchase_%'] = 100 - feb_s['purchase_rate_%']
feb_s['overall_rate_%'] = (feb_s['purchase'] / feb_s['view']) * 100

print("\n--- February ---")
print(feb_s)

jan_values = [jan_s['view'], jan_s['cart'], jan_s['purchase']]
feb_values = [feb_s['view'], feb_s['cart'], feb_s['purchase']]


plt.figure(figsize=(8, 4))
plt.plot(['View', 'Cart', 'Purchase'], jan_values, marker='o', label='Yanvar')
plt.plot(['View', 'Cart', 'Purchase'], feb_values, marker='o', label='Fevral')

plt.title("Number of Users during Stages (Funnel)")
plt.ylabel(" Number of Users")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# cart rate, purchase rate, overall rate and top categories in january

df_cat_jan = df_jan_clean.groupby(['category_code', 'event_type'])['user_id'].nunique().unstack().fillna(0)

df_cat_jan['cart_rate_%'] = (df_cat_jan['cart'] / df_cat_jan['view']) * 100
df_cat_jan['purchase_rate_%'] = (df_cat_jan['purchase'] / df_cat_jan['cart']) * 100
df_cat_jan['overall_rate_%'] = (df_cat_jan['purchase'] / df_cat_jan['view']) * 100

display(df_cat_jan.sort_values(by='view', ascending=False))

top_jan = df_cat_jan.sort_values(by='view', ascending=True).tail(8)

plt.figure(figsize=(10, 5))
plt.barh(top_jan.index, top_jan['view'], color='#4C72B0')
plt.title("January - Top Categories by Views", fontsize=14, fontweight='bold')
plt.xlabel("Views", fontsize=11)
plt.ylabel("Category Code", fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
# cart rate, purchase rate, overall rate and top categories in february

df_cat_feb = df_feb_clean.groupby(['category_code', 'event_type'])['user_id'].nunique().unstack().fillna(0)

df_cat_feb['cart_rate_%'] = (df_cat_feb['cart'] / df_cat_feb['view']) * 100
df_cat_feb['purchase_rate_%'] = (df_cat_feb['purchase'] / df_cat_feb['cart']) * 100
df_cat_feb['overall_rate_%'] = (df_cat_feb['purchase'] / df_cat_feb['view']) * 100

display(df_cat_feb.sort_values(by='view', ascending=False))

top_feb = df_cat_feb.sort_values(by='view', ascending=True).tail(8)

plt.figure(figsize=(10, 5))
plt.barh(top_feb.index, top_feb['view'], color='#DD8452')
plt.title("February - Top Categories by Views", fontsize=14, fontweight='bold')
plt.xlabel("Views", fontsize=11)
plt.ylabel("Category Code", fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# daily funnel for January

df_jan_clean['date'] = pd.to_datetime(df_jan_clean['event_time']).dt.date
df_daily_jan = df_jan_clean.groupby(['date', 'event_type'])['user_id'].nunique().unstack().fillna(0)

df_daily_jan['cart_rate_%'] = (df_daily_jan['cart'] / df_daily_jan['view']) * 100
df_daily_jan['purchase_rate_%'] = (
    df_daily_jan['purchase'] / df_daily_jan['cart']
) * 100
df_daily_jan['overall_rate_%'] = (
    df_daily_jan['purchase'] / df_daily_jan['view']
) * 100

display(df_daily_jan)

display(HTML('<div style="height: 20px;"></div>'))

plt.figure(figsize=(12, 5))
plt.plot(
    df_daily_jan.index,
    df_daily_jan['cart_rate_%'],
    label='Cart Rate (%)',
    marker='o',
)
plt.plot(
    df_daily_jan.index,
    df_daily_jan['purchase_rate_%'],
    label='Purchase Rate (%)',
    marker='s',
)
plt.plot(
    df_daily_jan.index,
    df_daily_jan['overall_rate_%'],
    label='Overall Rate (%)',
    marker='^',
)

plt.title('January - Daily Conversion Rates', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=11)
plt.ylabel('Rate (%)', fontsize=11)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(HTML('<div style="height: 30px;"></div>'))

In [ ]:
# daily funnel for February

df_feb_clean['date'] = pd.to_datetime(df_feb_clean['event_time']).dt.date
df_daily_feb = df_feb_clean.groupby(['date', 'event_type'])['user_id'].nunique().unstack().fillna(0)

df_daily_feb['cart_rate_%'] = (df_daily_feb['cart'] / df_daily_feb['view']) * 100
df_daily_feb['purchase_rate_%'] = (
    df_daily_feb['purchase'] / df_daily_feb['cart']
) * 100
df_daily_feb['overall_rate_%'] = (
    df_daily_feb['purchase'] / df_daily_feb['view']
) * 100

display(df_daily_feb)

display(HTML('<div style="height: 20px;"></div>'))

plt.figure(figsize=(12, 5))
plt.plot(
    df_daily_feb.index,
    df_daily_feb['cart_rate_%'],
    label='Cart Rate (%)',
    marker='o',
)
plt.plot(
    df_daily_feb.index,
    df_daily_feb['purchase_rate_%'],
    label='Purchase Rate (%)',
    marker='s',
)
plt.plot(
    df_daily_feb.index,
    df_daily_feb['overall_rate_%'],
    label='Overall Rate (%)',
    marker='^',
)

plt.title('February - Daily Conversion Rates', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=11)
plt.ylabel('Rate (%)', fontsize=11)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(HTML('<div style="height: 30px;"></div>'))

In [ ]:
# monthly comparison

monthly_comparison = pd.DataFrame({
    'January': jan_s,
    'February': feb_s
}).fillna(0)

monthly_comparison['Improvement (Diff)'] = monthly_comparison['February'] - monthly_comparison['January']

display(monthly_comparison)

In [ ]:
# time between view and purchase

df_jan_clean['event_time'] = pd.to_datetime(df_jan_clean['event_time'])

first_view = df_jan_clean[df_jan_clean['event_type'] == 'view'].groupby('user_id')['event_time'].min()
first_purchase = df_jan_clean[df_jan_clean['event_type'] == 'purchase'].groupby('user_id')['event_time'].min()

conversion_time = pd.DataFrame({'first_view': first_view, 'first_purchase': first_purchase}).dropna()
conversion_time['time_diff'] = conversion_time['first_purchase'] - conversion_time['first_view']

avg_time = conversion_time['time_diff'].mean()
print(f"Average time (View -> Purchase): {avg_time}")

display(HTML('<div style="height: 15px;"></div>'))

plt.figure(figsize=(10, 5))
plot_data = conversion_time[conversion_time['hours_diff'] <= 168]['hours_diff']

plt.hist(plot_data, bins=30, color='#4C72B0', edgecolor='black', alpha=0.8)
plt.title("January - Time Difference Between First View and Purchase (Hours)", fontsize=14, fontweight='bold')
plt.xlabel("Hours", fontsize=11)
plt.ylabel("Number of Users", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# high view low conversion products

prod_funnel = df_jan_clean.groupby(['product_id', 'event_type'])['user_id'].nunique().unstack().fillna(0)

if 'view' in prod_funnel.columns and 'purchase' in prod_funnel.columns:
    prod_funnel['conversion_rate_%'] = (prod_funnel['purchase'] / prod_funnel['view']) * 100
    high_view_low_conv = prod_funnel.sort_values(by='view', ascending=False)

    display(high_view_low_conv.head(20))

    top_20 = high_view_low_conv.head(20).reset_index()

    plt.figure(figsize=(12, 6))
    plt.bar(top_20['product_id'].astype(str), top_20['view'], color='#4C72B0', alpha=0.8, label='Views')

    ax2 = plt.twinx()
    ax2.plot(top_20['product_id'].astype(str), top_20['conversion_rate_%'], color='#DD8452', marker='o', linewidth=2, label='Conversion Rate (%)')

    plt.title("January - Top 20 Viewed Products & Conversion Rates", fontsize=14, fontweight='bold')
    plt.xlabel("Product ID", fontsize=11)
    plt.ylabel("Views", fontsize=11)
    ax2.set_ylabel("Conversion Rate (%)", fontsize=11)

    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()


In [ ]:
# concatenating of datasets

df = pd.concat([df_jan, df_feb], ignore_index=True)

# --- 2. TOP 8 CATEGORIES FUNNEL ---
top_8_cats = df_clean[df_clean['event_type'] == 'view']['category_code'].value_counts().head(8).index

cat_df = df_clean[df_clean['category_code'].isin(top_8_cats)]
cat_funnel = cat_df.groupby(['category_code', 'event_type'])['user_id'].nunique().unstack().fillna(0)

cat_funnel['view_to_cart_%'] = (cat_funnel['cart'] / cat_funnel['view']) * 100
cat_funnel['cart_to_purchase_%'] = (cat_funnel['purchase'] / cat_funnel['cart']) * 100
cat_funnel['overall_rate_%'] = (cat_funnel['purchase'] / cat_funnel['view']) * 100

print("--- TOP 8 CATEGORIES FUNNEL ---")
display(cat_funnel.sort_values(by='view', ascending=False))

top_low_conv = high_view_low_conv.head(20).reset_index()

    plt.figure(figsize=(12, 6))
    plt.barh(top_low_conv['product_id'].astype(str), top_low_conv['conversion_rate_%'], color='#DD8452')
    plt.title("Top High-View Products with Lowest Conversion Rate", fontsize=14, fontweight='bold')
    plt.xlabel("Conversion Rate (%)", fontsize=11)
    plt.ylabel("Product ID", fontsize=11)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()


In [ ]:
#  TOP 10 BRANDS BY PURCHASE & VIEW-TO-PURCHASE RATE ---

top_10_brand_names = df_clean[df_clean['event_type'] == 'purchase']['brand'].value_counts().head(10).index

brand_df = df_clean[df_clean['brand'].isin(top_10_brand_names)]
brand_counts = brand_df.groupby(['brand', 'event_type'])['user_id'].nunique().unstack().fillna(0)

brand_counts['view_to_purchase_%'] = (brand_counts['purchase'] / brand_counts['view']) * 100
top_10_brands = brand_counts.loc[top_10_brand_names]

display(top_10_brands[['view', 'purchase', 'view_to_purchase_%']])
display(HTML('<div style="height: 15px;"></div>'))

plt.figure(figsize=(10, 6))
plt.barh(top_10_brands.index[::-1], top_10_brands['view_to_purchase_%'][::-1], color='#55A868', alpha=0.8)
plt.title("Top 10 Brands by View-to-Purchase Rate (%)", fontsize=14, fontweight='bold')
plt.xlabel("View-to-Purchase Rate (%)", fontsize=11)
plt.ylabel("Brand", fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# PRICE OUTLIERS USING IQR (PURCHASE EVENTS) ---
purchases = df_clean[df_clean['event_type'] == 'purchase']['price'].dropna()

q1 = purchases.quantile(0.25)
q3 = purchases.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = purchases[(purchases < lower_bound) | (purchases > upper_bound)]

print("\n--- PRICE OUTLIERS (PURCHASE) ---")
print(f"Q1: {q1:.2f}, Q3: {q3:.2f}, IQR: {iqr:.2f}")
print(f"Alt sərhəd: {lower_bound:.2f}, Üst sərhəd: {upper_bound:.2f}")
print(f"Outlier qiymətlərin sayı: {len(outliers)}")